# Dual-Space Methods

Copyright (C) 2026 Andreas Kloeckner

<details>
<summary>MIT License</summary>
Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.
</details>

---

Based on [A Dual-space Multilevel Kernel-splitting Framework for Discrete and Continuous Convolution](https://arxiv.org/abs/2308.00292) (Shidong Jiang, Leslie Greengard)


In [86]:
import numpy as np
import numpy.linalg as la
from scipy.special import erf, erfc
import matplotlib.pyplot as plt
from sumpy.visualization import FieldPlotter

# a lot of dividing by zero here, silence warnings
np.errstate(divide="ignore", invalid="ignore").__enter__()

rng = np.random.default_rng(230800292)

def _at_zero(r: np.ndarray, value: float, expression: np.ndarray) -> np.ndarray:
    """Replace the removable r=0 value in a radial-kernel expression."""
    return np.where(np.asarray(r) == 0.0, value, expression)

# The diameter of [-1/2, 1/2]^3, as in Lemma 8.
BOX_DIAMETER = np.sqrt(3.0)

### A refresher: `erf` and `erfc`

In [87]:
x = np.linspace(-6, 6, 100)
plt.plot(x, erf(x), label="erf")
plt.plot(x, erfc(x), label="erfc")
plt.plot(x, erf(x) + erfc(x), label="sum")
plt.legend()

### Decomposing the Green's function

In [88]:
# change me
sigma = 1

def M(r):
    return 1/r * erf(r/sigma) 
def W(r):
    return 1/r * erfc(r/sigma) 

In [89]:
r = np.linspace(0, 5, 1000)

plt.plot(r, 1/r, "-", label="1/r")
plt.plot(r, M(r), label="M(r): mollified, erf")
plt.plot(r, W(r), label="R(r): residual, erfc")

del sigma

plt.legend(loc="best")
plt.ylim([0, 10])

Observe decay past $6 \sigma$:

In [90]:
r = np.linspace(6, 10, 100)

plt.semilogy(r, erfc(r))

In [91]:
class FourierQuadrature:
    def __init__(self, *, period, n):
        self.h = h = 2.0 * np.pi / period
        self.modes = modes = np.arange(-n, n + 1)
        self.kx, self.ky, self.kz = np.meshgrid(h * modes, h * modes, h * modes, indexing="ij")
        self.k = np.column_stack((self.kx.ravel(), self.ky.ravel(), self.kz.ravel()))
        self.weight = (h / (2.0 * np.pi)) ** 3 
        self.k_mag = np.linalg.norm(self.k, axis=1) 
        
    def __call__(self, targets, k_values):
        return np.real(np.exp(1j * targets @ self.k.T) @ (self.weight * k_values))
    
    def plot(self, values, z_mode=0):
        z_idx, = np.where(self.modes == z_mode)
        n_modes = len(self.modes)
        
        myslice =  values.reshape(n_modes, n_modes, n_modes)[:, :, z_idx] 
        
        ax = plt.subplot(121)
        plt.imshow(np.log10(1e-15 + np.abs(myslice)))
        plt.colorbar()
        ax = plt.subplot(122)
        plt.imshow(np.arctan2(myslice.imag, myslice.real))

fquad = FourierQuadrature(period=8, n=48)

In [92]:
def M_hat(k_mag):
    return 4*np.pi * np.exp(-(sigma * k_mag) ** 2 / 4.0) / k_mag**2

sigma = 1
m_hat_values = M_hat(fquad.k_mag) 
del sigma

fquad.plot(m_hat_values)

la.norm(m_hat_values, np.inf)

### Dealing with the Singularity

In [93]:
def windowed(r, *, sigma, b) -> np.ndarray:
    """Physical-space W, evaluated stably at r=0."""
    a = BOX_DIAMETER + b * sigma
    value = (erf(r / sigma) - 0.5 * erf((a + r) / sigma)
             + 0.5 * erf((a - r) / sigma)) / r

    # Differentiate the numerator at zero.  The two large terms cancel here.
    limit = (2.0 / (np.sqrt(np.pi) * sigma)) * (1.0 - np.exp(-(a / sigma) ** 2))
    return _at_zero(r, limit, value)

r = np.linspace(0, 10, 1000)
plt.ylim([0, 2])

sigma = 1
mvals = 1/r * erf(r/sigma) 
wvals =windowed(r, sigma=sigma, b=6) 
plt.plot(r, 1/r, label="1/r")
plt.plot(r, mvals, label="M(r): mollified, erf")
plt.plot(r, wvals, label="W(r): windowed")
del sigma

plt.legend()

In [94]:
def windowed_hat(k_mag, sigma, c_tilde):
    value = (8.0 * np.pi * (np.sin(c_tilde * k_mag / 2.0) / k_mag) ** 2  
             * np.exp(-(sigma * k_mag) ** 2 / 4.0))
    return _at_zero(k_mag, 2.0 * np.pi * c_tilde**2, value)

sigma = 0.2; b = 6
w_hat_values = windowed_hat(fquad.k_mag, sigma=sigma, c_tilde=BOX_DIAMETER + b * sigma)
del sigma
del b

fquad.plot(w_hat_values)

In [95]:
fp = FieldPlotter(
    center=np.zeros(3),
    extent=np.array([1, 1, 0]),
    npoints=(10, 10, 1))
targets = fp.points.T
r = la.norm(targets, 2, axis=1)

sigma = 0.2; b = 0.6
err = fquad(targets, windowed_hat(fquad.k_mag, sigma=sigma, c_tilde=BOX_DIAMETER + b * sigma)) - windowed(r, sigma=sigma, b=b)
del sigma
del b

la.norm(err, np.inf)

### Multiple sources

In [96]:
sources = np.array([[0, 0, 0]], dtype=np.float64)

fp = FieldPlotter(
    center=np.zeros(3),
    extent=np.array([1, 1, 0]),
    npoints=(100, 100, 1))
targets = fp.points.T

def pairwise_potential(targets: np.ndarray, sources: np.ndarray,
                       charges: np.ndarray, kernel) -> np.ndarray:
    distances = la.norm(targets[:, None, :] - sources[None, :, :], axis=2)
    return kernel(distances) @ charges

pot = pairwise_potential(targets, sources, [1], lambda r: 1/r)

fp.show_scalar_in_matplotlib(np.log10(1e-15 + np.abs(pot)))
plt.colorbar()

In [97]:
def fourier_window_potential(targets: np.ndarray, sources: np.ndarray,
                             charges: np.ndarray, sigma: float, b: float) -> np.ndarray:
    source_phase = np.exp(-1j * fquad.k @ sources.T) @ charges
    return fquad(targets,
        windowed_hat(fquad.k_mag, sigma, BOX_DIAMETER + b * sigma) 
        * source_phase)

sigma = 0.18; b = 6
sources = rng.uniform(-0.5, 0.5, size=(12, 3))
targets = rng.uniform(-0.5, 0.5, size=(10, 3))
charges = rng.normal(size=12)
exact = pairwise_potential(targets, sources, charges,
                           lambda radius: windowed(radius, sigma=sigma, b=b))

spectral = fourier_window_potential(targets, sources, charges, sigma=sigma, b=b)
quadrature_error = np.max(np.abs(spectral - exact)) / np.max(np.abs(exact))
print(quadrature_error)
del sigma
del b

In [98]:
fquad.k.shape